In [ ]:
%load_ext autoreload
%autoreload 2

In [160]:
import sys
import os
sys.path.append("/home/pwiesenbach/BertGCN")
os.chdir("/home/pwiesenbach/BertGCN")

In [210]:
import glob
import pathlib 
import shutil
from pathlib import Path
import re

In [120]:
letter_path = "outputs/letters"
#unklar_letter_path = "outputs/letters/testunklar/"   
annotations = "/prj/doctoral_letters/MIEdeep/corpus/webanno_proj/2022-05-24_all_curated_inclmedind/annotation"

In [131]:
anns = glob.glob(str(Path(annotations) / "**" / "*.tsv"))
anns[:2]

['/prj/doctoral_letters/MIEdeep/corpus/webanno_proj/2022-05-24_all_curated_inclmedind/annotation/BR3.0000000000000010042787341.01.000.txt_0/kiriakouchristina.tsv',
 '/prj/doctoral_letters/MIEdeep/corpus/webanno_proj/2022-05-24_all_curated_inclmedind/annotation/BR3.0000000000000010048512920.01.000.txt_0/kiriakouchristina.tsv']

In [209]:
letters = glob.glob(str(Path(letter_path) / "**" / "*.txt" ), recursive=True)
letters[:2]

['outputs/letters/3/1594/Blutdrucksenker_unklar-Blutdrucksenker_Blutdruck.txt',
 'outputs/letters/3/1594/1829/Blutdrucksenker_Herzschw.txt']

In [191]:
pairs = dict()
for l in letters:
    with open(l, "r") as f:
        txt = f.read()
        txt_aufnahme, txt_geb = re.findall("\d{4} - \d{2} - \d{2}", txt)
        txt_aufnahme = txt_aufnahme.replace(" ", "")
        txt_geb = txt_geb.replace(" ", "")
        print(l, txt_aufnahme, txt_geb)
    
    for a in anns:
        if "test2" in a or "dummy" in a:
            continue
        with open(a, "r") as f:
            ann = f.readlines()
            try:
                ann_aufnahme = [x.strip() for x in ann if x.startswith("#Text=Aufnahmedatum:")][0]
            except:
                print(a)
                raise
            ann_aufnahme = re.search("\d{4}-\d{2}-\d{2}", ann_aufnahme).group(0)
            ann_geb = [x.strip() for x in ann if x.startswith("#Text=Geburtsdatum:")][0]
            ann_geb = re.search("\d{4}-\d{2}-\d{2}", ann_geb).group(0)
            assert aufnahme != [] and geb != []
        if txt_aufnahme == ann_aufnahme and txt_geb == ann_geb:
            pairs[l] = a
            found = True
            break
    if not found:
        raise Exception(l, txt_geb, txt_aufnahme)
    #break
len(pairs)

outputs/letters/3/1594/Blutdrucksenker_unklar-Blutdrucksenker_Blutdruck.txt 2030-01-14 1956-05-06
outputs/letters/3/1594/1829/Blutdrucksenker_Herzschw.txt 2027-01-23 1953-03-05
outputs/letters/3/1594/2201/DM_nur Insulin.txt 2510-05-14 2126-05-05
outputs/letters/3/2465/DM_nur Tabletten-DM_nur Tabletten.txt 2021-05-21 1950-03-14
outputs/letters/3/2465/554/Cholesterinsenker_KHK.txt 2471-12-28 2114-01-05
outputs/letters/3/2465/498/Cholesterinsenker_KHK.txt 2025-08-05 1954-08-16
outputs/letters/3/2465/471/DM_Insulin und Tabletten.txt 2022-04-22 1944-10-29
outputs/letters/3/2465/2162/Cholesterinsenker_KHK.txt 2028-05-10 1974-07-26
outputs/letters/3/37/Blutdrucksenker_unklar-Blutdrucksenker_Blutdruck.txt 2026-07-21 1963-11-16
outputs/letters/3/37/1829/Blutdrucksenker_Herzschw.txt 2027-01-23 1953-03-05
outputs/letters/3/2573/Blutdrucksenker_unklar-Blutdrucksenker_Blutdruck.txt 2026-11-09 1968-11-24
outputs/letters/3/2573/1466/Blutdrucksenker_beides.txt 2035-01-23 1974-01-20
outputs/letters/3/2

68

In [189]:
l = [pathlib.PurePath(x[1]).parent.name.rsplit(".", 1)[0] for x in pairs]

for ll in l:
    print(ll, end="; ")

BR3.0000000000000010075612266.01.000(.doc|.docx); BR3.0000000000000010065433942.01.000(.doc|.docx); BR3.0000000000000010072083238.01.000(.doc|.docx); BR3.0000000000000010018282759.01.000(.doc|.docx); BR3.0000000000000010030609171.01.000(.doc|.docx); BR3.0000000000000010037557908.01.000(.doc|.docx); BR3.0000000000000010022551988.01.000(.doc|.docx); BR3.0000000000000010065524628.01.000(.doc|.docx); BR3.0000000000000010062294144.01.000(.doc|.docx); BR3.0000000000000010065433942.01.000(.doc|.docx); BR3.0000000000000010037746437.01.000(.doc|.docx); BR3.0000000000000010085655036.01.000(.doc|.docx); BR3.0000000000000010022763300.01.000(.doc|.docx); BR3.0000000000000010020433297.01.000(.doc|.docx); BR3.0000000000000010020781472.01.000(.doc|.docx); BR3.0000000000000010014147321.01.000(.doc|.docx); BR3.0000000000000010012955953.01.000(.doc|.docx); BR3.0000000000000010071814937.01.000(.doc|.docx); BR3.0000000000000010014381301.01.000(.doc|.docx); BR3.0000000000000010034504498.01.000(.doc|.docx); 

In [217]:
orig_dir ="outputs/letters/original"

for l in letters:
    orig_name = pathlib.PurePath(pairs[l]).parent.name.rsplit(".", 1)[0]
    orig_files = glob.glob(str(Path(orig_dir) / (orig_name + "*")))
    assert orig_files != []
    _orig_files = [x for x in orig_files if ".docx" in x]
    if _orig_files == []:
        _orig_files = [x for x in orig_files if ".doc" in x]
    orig_file = _orig_files[0]
    print(pathlib.PurePath(l).parent, orig_file)
    shutil.copy(orig_file, pathlib.PurePath(l).parent)
    

outputs/letters/3/1594 outputs/letters/original/BR3.0000000000000010075612266.01.000.doc
outputs/letters/3/1594/1829 outputs/letters/original/BR3.0000000000000010065433942.01.000.docx
outputs/letters/3/1594/2201 outputs/letters/original/BR3.0000000000000010072083238.01.000.docx
outputs/letters/3/2465 outputs/letters/original/BR3.0000000000000010018282759.01.000.docx
outputs/letters/3/2465/554 outputs/letters/original/BR3.0000000000000010030609171.01.000.docx
outputs/letters/3/2465/498 outputs/letters/original/BR3.0000000000000010037557908.01.000.docx
outputs/letters/3/2465/471 outputs/letters/original/BR3.0000000000000010022551988.01.000.docx
outputs/letters/3/2465/2162 outputs/letters/original/BR3.0000000000000010065524628.01.000.docx
outputs/letters/3/37 outputs/letters/original/BR3.0000000000000010062294144.01.000.docx
outputs/letters/3/37/1829 outputs/letters/original/BR3.0000000000000010065433942.01.000.docx
outputs/letters/3/2573 outputs/letters/original/BR3.000000000000001003774